# Setup

In [1]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

The agentic loop keeps calling the model until it stops by using a `while` loop and a flag called `has_function_calls`.

Here's how it works:

1.  **Initial Call:** The loop starts by calling the model with the initial instructions, the user's question, and available tools.
2.  **Process Response:** The model's response is processed.
    *   If the response contains a `function_call` (meaning the model wants to use a tool, like `search`), the `has_function_calls` flag is set to `True`. The tool is executed, and its output is appended to the message history.
    *   If the response contains a `message` (a textual answer), it is printed, and the `has_function_calls` flag remains `False` (or is reset if it was previously `True` from processing a function call).
3.  **Memory Update:** All outputs from the current turn (model messages and tool call outputs) are appended to the `messages` list, which serves as the agent's memory. This ensures the model sees its previous actions and their res

In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

# Q1. First trace

Wrap the `rag()` method so each call produces a span. The simplest way
is to create a `RAGTraced` subclass of `RAGBase` that wraps `rag()`,
`search()`, and `llm()` each in their own span.

Run this query:

> How does the agentic loop keep calling the model until it stops?

The console exporter prints every finished span as a dictionary.
Count the spans in the console output - each one is a separate
`ReadableSpan` entry. How many spans does the trace produce?

* 1
* 3
* 5
* 7

In [3]:
from starter import client, index
from rag_helper import RAGBase
import time

def calc_price(usage):
    input_price_per_million = 0.75
    output_price_per_million = 4.50

    input_cost = (usage.prompt_tokens / 1_000_000) * input_price_per_million
    output_cost = (usage.completion_tokens / 1_000_000) * output_price_per_million
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

class RAGTraced(RAGBase):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            results = self.index.search(query, num_results=num_results)

            span.set_attribute("query", query)
            span.set_attribute("num_results", num_results)
            
            return results

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            #start_time = time.time()
            input_messages = [
                {'role': 'developer', 'content': self.instructions},
                {'role': 'user', 'content': prompt}
            ]
    
            response = self.llm_client.chat.completions.create(
                model=self.model,
                messages=input_messages
            )
    
            output_message = response.choices[0].message.content

            usage_results = calc_price(response.usage)

            span.set_attribute("prompt_tokens", response.usage.prompt_tokens)
            span.set_attribute("completition_tokens", response.usage.completion_tokens)
            span.set_attribute("total_cost", usage_results["total_cost"])
    
            return response.choices[0].message.content

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            search_results = self.search(query)
            prompt = self.build_prompt(query, search_results)
            answer = self.llm(prompt)

            span.set_attribute("query", query)
            span.set_attribute("prompt", prompt)
            span.set_attribute("answer", answer)
            
            return answer

In [4]:
rag = RAGTraced(index=index, llm_client=client)

In [5]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0x78c5afcbe2315050853c8cc1d1b55e35",
        "span_id": "0x7d2f927cd9017b59",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xbcc9682d20305782",
    "start_time": "2026-07-21T05:14:22.610396Z",
    "end_time": "2026-07-21T05:14:22.614969Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "bcc5c77e-ce27-4732-a631-e20dbb4296d2",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x78c5afcbe23150508

**Answer:** 3

### Q2. Capturing metrics as span attributes

Spans are not just timing markers - you can attach any information you
want to them with `set_attribute`. We already use spans to record how
long each step takes. Now we'll add the metrics we care about: tokens
and cost.

Read the token usage from the LLM response (the `llm()` method in the
starter already returns the raw response object) and set them as
attributes on the `llm` span:

```python
span.set_attribute("input_tokens", usage.input_tokens)
span.set_attribute("output_tokens", usage.output_tokens)
```

And since we know both input and output tokens, we can also compute
the cost using the code from the previous modules.

Now re-run the query. How many input tokens do we see?

* 700
* 7000
* 70000
* 700000

> These numbers vary between runs. Pick the closest option.

**Answer:** 7000

# Q3. Span timing

Each span automatically records its duration. Look at the console output
from Q1 and find the durations for the `search` span and the `llm` span.

For a typical query, roughly how long does the LLM call take?

* Under 100ms
* 100-500ms
* 500-2000ms
* Over 2000ms

> The first call can be slower (cold start). Pick the range you see
> most often

**Answer:** Over 2000ms

# Q4. Saving traces to SQLite

Right now the spans are printed to the terminal and then gone. We don't
save them.

We want to persist them so we can query them later.

In this homework, we'll use SQLite - it's a more lightweight option than
Postgres, so we don't need to set up any docker containers in this homework.

Our instrumentation is already done, we don't need to change anything there.
But we need to create a custom exporter. Instead of printing the spans,
it will save them to the database.

OTel calls the exporter through the same span processor we already use,
we just swap the destination.

Now we will create a custom exporter that saves each finished span to a
SQLite database. The exporter extends `SpanExporter`. It has the following methods:

- `export` method that receives a list of `ReadableSpan` objects
- `shutdown` and `force_flush` methods

Let's implement it:

```python
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True
```

Replace the console exporter with this new exporter:

```python
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
```

Re-run the query from Q1. Which span names appear in the `spans` table?

* Only `rag`
* `rag` and `llm`
* `rag`, `search`, and `llm`
* `search`, `llm`, and `judge`

In [6]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("prompt_tokens"),
                    attrs.get("completition_tokens"),
                    attrs.get("total_cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [7]:
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

In [15]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)

{
    "name": "search",
    "context": {
        "trace_id": "0xefb3ac7423aacfe684b2481451662ab9",
        "span_id": "0x405c1bba2efcb7bd",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x5773d099e19bc89d",
    "start_time": "2026-07-21T05:16:23.864483Z",
    "end_time": "2026-07-21T05:16:23.867186Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "bcc5c77e-ce27-4732-a631-e20dbb4296d2",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xefb3ac7423aacfe68

In [16]:
conn = sqlite3.connect("traces.db")
cursor = conn.execute("SELECT * FROM spans")
results = cursor.fetchall()
for row in results:
    print(row)

('search', 1784610891916486400, 1784610891919204100, None, None, None)
('llm', 1784610891936159300, 1784610897506354100, 7933, 452, 0.00798375)
('rag', 1784610891916415000, 1784610897529996900, None, None, None)
('search', 1784610912306753200, 1784610912310149700, None, None, None)
('llm', 1784610912319106900, 1784610916755614000, 7933, 438, 0.00792075)
('rag', 1784610912306677900, 1784610916772654200, None, None, None)
('search', 1784610936924008700, 1784610936926888700, None, None, None)
('llm', 1784610936934753400, 1784610941946959200, 7933, 351, 0.00752925)
('rag', 1784610936923951200, 1784610941955294300, None, None, None)
('search', 1784610983864482800, 1784610983867186000, None, None, None)
('llm', 1784610983882015700, 1784610989868480100, 7933, 485, 0.00813225)
('rag', 1784610983864417500, 1784610989886585200, None, None, None)


**Answer:** rag, search, and llm

# Q5. Querying trace data

The traces are now in SQLite. Run one more query through the traced
RAG, then query the database.

The `rag` span wraps everything, so its duration includes both
`search` and `llm`. To see where time actually goes, exclude the
`rag` span and compare the children.

Using SQL (or pandas), compute the total duration for each span name
excluding `rag`. Which span type takes the most total time?

* `search`
* `llm`
* They're all about the same

In [17]:
import pandas as pd

df = pd.read_sql_query("""
        SELECT name, (end_time - start_time) / 1000000000.0 AS total_duration_seg 
        FROM spans
        WHERE name <> 'rag'
        ORDER BY total_duration_seg DESC
""", conn)
print(df)

     name  total_duration_seg
0     llm            5.986464
1     llm            5.570195
2     llm            5.012206
3     llm            4.436507
4  search            0.003397
5  search            0.002880
6  search            0.002718
7  search            0.002703


**Answer:** llm

# Q6. Token stability across runs

Load the SQLite data with pandas. One thing a dashboard can tell you
is how stable your system is. If the same query always produces the
same number of input tokens, the context your RAG retrieves is
consistent. If it varies a lot, something in the search may be
unstable.

Run the same query from Q1 three more times (so you have 4 RAG calls
total in the database). Then compute the input tokens for each `llm`
span.

How much do the input tokens vary across these 4 runs?

* They're identical
* Within 10% of each other
* Within 50% of each other
* They vary more than 50%

In [18]:
df = pd.read_sql_query("""
        SELECT name, input_tokens
        FROM spans
        WHERE name = 'llm'
""", conn)
print(df)

  name  input_tokens
0  llm          7933
1  llm          7933
2  llm          7933
3  llm          7933


**Answer:** They're identical